# PART 1: LLM inference
We have a trained model and want to use it. Includes inference examples using HuggingFace transformers, vLLM, ollama.

In the practical part, test the code, modify the decoding hyperparameters, and the used models, and test how well existing models (off-the-shelf) perform at your task.

**Warning**: Installing vLLM takes a few minutes. If Colab asks you to **restart the session** afterwards, do so and continue with the next cell.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
%pip install -q -U transformers accelerate openai datasets scikit-learn vllm

name, memory.total [MiB]
Tesla T4, 15360 MiB
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, b

In [2]:
import gc, time, json, subprocess, requests
import torch, pandas as pd
from datasets import load_dataset

BF16 = torch.cuda.get_device_capability()[0] >= 8      # T4 (Colab) has no bfloat16 -> float16
DTYPE = torch.bfloat16 if BF16 else torch.float16
print(torch.cuda.get_device_name(0), "| dtype:", DTYPE)

Tesla T4 | dtype: torch.float16


In [3]:
from transformers import pipeline

HF_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
generator = pipeline("text-generation", model=HF_MODEL, dtype=DTYPE, device_map="cuda")

messages = [{"role": "user", "content": "Explain in one sentence what a language model does."}]
out = generator(messages, max_new_tokens=60, do_sample=False)
print(out[0]["generated_text"][-1]["content"])

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

A language model is an artificial intelligence system designed to simulate human-like language processing and generation capabilities, enabling it to understand, generate, and respond to natural language inputs.


The following is what the pipeline does for you under the hood. The **chat template** turns the list of messages into the single string the model was trained on. Each model may have a different chat template.

In [4]:
tok = generator.tokenizer
model = generator.model

messages = [{"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Is the review 'A dull, lifeless film.' positive or negative?"}]

print(tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
Is the review 'A dull, lifeless film.' positive or negative?<|im_end|>
<|im_start|>assistant



In [5]:
inputs = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
print("prompt tokens:", inputs["input_ids"].shape[1])

output = model.generate(**inputs, max_new_tokens=20, do_sample=False)
new_tokens = output[0, inputs["input_ids"].shape[1]:]                 # keep only the generated part
print(tok.decode(new_tokens, skip_special_tokens=True))

prompt tokens: 34
The review "A dull, lifeless film" is clearly negative. The term "dull"


**What the model actually computes: next-token probabilities**

At every step the model gives a probability to **every** token in its vocabulary. Temperature `T` rescales the scores before the softmax: low `T` sharpens, high `T` flattens.

In [6]:
@torch.no_grad()
def next_token_table(text, temperatures=(0.5, 1.0, 2.0), k=8):
    ids = tok(text, return_tensors="pt").to(model.device)
    logits = model(**ids).logits[0, -1].float()                 # scores for the next token
    top = torch.topk(logits, k).indices
    table = {"token": [repr(tok.decode(int(i))) for i in top]}
    for T in temperatures:
        p = torch.softmax(logits / T, dim=-1)
        table[f"p (T={T})"] = [round(p[i].item(), 3) for i in top]
    return pd.DataFrame(table)

next_token_table("The delivery was late and the box was")

,token,p (T=0.5),p (T=1.0),p (T=2.0)
0,' damaged',0.445,0.138,0.010
1,' not',0.296,0.113,0.009
2,' broken',0.088,0.061,0.007
3,' missing',0.043,0.043,0.006
4,' in',0.027,0.034,0.005
5,' empty',0.017,0.027,0.005
6,' too',0.015,0.025,0.004
7,' left',0.009,0.019,0.004


## Decoding parameters

| Parameter | Effect |
|---|---|
| `do_sample=False` | greedy: always the most probable token (deterministic) |
| `temperature` | randomness of sampling (needs `do_sample=True`) |
| `top_p`, `top_k` | sample only among the most likely tokens |
| `num_beams` | beam search: keep several candidate sequences |
| `repetition_penalty` | discourage repeating tokens |
| `max_new_tokens` | hard limit on the output length |

In [7]:
prompt = [{"role": "user", "content": "Write a one-sentence review of a film about a lighthouse keeper."}]
inputs = tok.apply_chat_template(prompt, add_generation_prompt=True, return_tensors="pt").to(model.device)

settings = {
    "greedy":                   dict(do_sample=False),
    "beam search (4 beams)":    dict(do_sample=False, num_beams=4),
    "sampling T=0.7, top_p=0.9": dict(do_sample=True, temperature=0.7, top_p=0.9),
    "sampling T=1.5":           dict(do_sample=True, temperature=1.5),
    "T=1.5 + top_k=20":         dict(do_sample=True, temperature=1.5, top_k=20),
}
torch.manual_seed(0)
for name, kw in settings.items():
    out = model.generate(**inputs, max_new_tokens=40, **kw)
    print(f"{name:>28}: {tok.decode(out[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True)}")

                      greedy: "Under the watchful gaze of the towering lighthouse, a dedicated and resilient man navigates life's challenges with unwavering determination, embodying the spirit of perseverance and dedication that defines the character
       beam search (4 beams): "The Lighthouse Keeper" is a poignant and heartwarming film that tells the story of a lighthouse keeper who must navigate the treacherous waters of the Pacific Ocean to save the lives of his
   sampling T=0.7, top_p=0.9: "Stunningly immersive and emotionally resonant, 'Lighthouses' is a poignant exploration of the human connection to the natural world through the lens of a dedicated lighthouse keeper's perspective."
              sampling T=1.5: The lighthouse keeper, portrayed with great skill by actor Tom Hanks, takes on a complex moral quandary that tests his commitment to his family and community as well as the principles he holds dear through
            T=1.5 + top_k=20: "Guarding the light for generatio

## Batching

Several prompts at once are much faster than a loop. Decoder models need **left padding** for batched generation.

In [8]:
reviews = ["A dull, lifeless film.", "An absolute joy from start to finish.", "Great cast, terrible script."]
batch = [[{"role": "user", "content": f"Answer with one word, positive or negative: {r}"}] for r in reviews]

generator.tokenizer.padding_side = "left"
outs = generator(batch, max_new_tokens=3, do_sample=False, batch_size=8)
for r, o in zip(reviews, outs):
    print(f"{o[0]['generated_text'][-1]['content']:>10}  <- {r}")

  Negative  <- A dull, lifeless film.
  Positive  <- An absolute joy from start to finish.
  Negative  <- Great cast, terrible script.


In [9]:
# Free the GPU before starting a server
del generator, model
gc.collect(); torch.cuda.empty_cache()

## **vLLM**: an inference server in the background

vLLM serves a model behind the **OpenAI-compatible API** and batches incoming requests on the GPU. In Colab we start it as a **background process**, wait until it answers, and talk to it from Python. The same code works against a GPU server, Ollama or a hosted API.

In [10]:
def start_background(cmd, log_file, health_url, timeout=900):
    # Start a server process in the background and wait until health_url responds.
    log = open(log_file, "w")
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    t0 = time.time()
    while time.time() - t0 < timeout:
        if proc.poll() is not None:                       # the process died: show the end of its log
            raise RuntimeError(open(log_file).read()[-3000:])
        try:
            if requests.get(health_url, timeout=2).ok:
                print(f"ready after {time.time() - t0:.0f} s")
                return proc
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(5)
    raise TimeoutError(f"not ready after {timeout} s, see {log_file}")

def stop(proc):
    proc.terminate(); proc.wait(timeout=60)

In [11]:
# The first start downloads the model and takes 2–5 minutes.
VLLM_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"    # alternatives: "google/gemma-3-1b-it" (HF login), a GaMS chat model from hf.co/cjvt

vllm_proc = start_background(
    ["vllm", "serve", VLLM_MODEL,
     "--port", "8000",
     "--dtype", "bfloat16" if BF16 else "half",
     "--max-model-len", "2048",               # small context = less memory
     "--gpu-memory-utilization", "0.6",
     "--enforce-eager"],                      # skip CUDA-graph capture: faster start-up on Colab
    log_file="vllm.log", health_url="http://localhost:8000/health")

ready after 100 s


**Calling the server**

In [12]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8000/v1", api_key="EMPTY")   # hosted API: its URL and your key

r = client.chat.completions.create(
    model=VLLM_MODEL,
    messages=[{"role": "user", "content": "Explain in one sentence what a language model does."}],
    temperature=0, max_tokens=60)
print(r.choices[0].message.content)
print(r.usage)                           # token counts: what a hosted API would bill

A language model is an artificial intelligence system designed to simulate human-like language processing and generation capabilities, enabling it to understand, generate, and respond to natural language inputs.
CompletionUsage(completion_tokens=34, prompt_tokens=40, total_tokens=74, completion_tokens_details=None, prompt_tokens_details=None)


In [13]:
# The same request without any client library: it is just HTTP + JSON
resp = requests.post("http://localhost:8000/v1/chat/completions", json={
    "model": VLLM_MODEL,
    "messages": [{"role": "user", "content": "Say hi in Slovenian."}],
    "max_tokens": 20})
print(json.dumps(resp.json()["choices"][0]["message"], indent=1, ensure_ascii=False))

{
 "role": "assistant",
 "content": "Vojnici! (Hello! in Slovenian)",
 "refusal": null,
 "audio": null,
 "function_call": null,
 "reasoning": null
}


**Decoding parameters through the API**

Similar knobs as before with API names: `temperature`, `top_p`, `max_tokens`, `stop`, `seed`, `n` (several answers).

In [14]:
prompt = [{"role": "user", "content": "Write a one-sentence review of a film about a lighthouse keeper."}]

for T in [0.0, 0.8, 1.3]:
    r = client.chat.completions.create(model=VLLM_MODEL, messages=prompt,
                                       temperature=T, top_p=0.95, max_tokens=40, seed=0,
                                       n=1 if T == 0 else 3)        # greedy gives one answer only
    print(f"--- temperature {T}")
    for c in r.choices:
        print("   ", c.message.content.strip())

--- temperature 0.0
    "Under the watchful gaze of the towering lighthouse, a dedicated and resilient man navigates life's challenges with unwavering determination, embodying the spirit of perseverance and dedication that defines the character
--- temperature 0.8
    " 'A Lighthouse Keeper' is a heartwarming tale of a dedicated and loyal lighthouse keeper who faces the challenges of life in a remote seaside town during the winter months. The script beautifully
    "Charming and emotionally resonant, 'The Lighthouse Keeper' offers an intimate look at the dedication and spirit required to serve as a lifeguard to a community, a lesson in resilience, and
    "Lost at Sea" is an atmospheric drama that weaves a haunting tale of courage, perseverance, and love as a lighthouse keeper faces a mysterious disappearance in a remote coastal town."
--- temperature 1.3
    " 'A Light Before Me' is a moving tribute to the life and sacrifices of an unworn lighthouse keeper, offering an engaging explor

In [15]:
# max_tokens and stop sequences cut the output
r = client.chat.completions.create(
    model=VLLM_MODEL, temperature=0, max_tokens=100, stop=["4."],
    messages=[{"role": "user", "content": "List the steps to make tea, numbered 1., 2., 3., ..."}])
print(r.choices[0].message.content)
print("finish_reason:", r.choices[0].finish_reason)

Sure! Here's a step-by-step guide on how to make tea:

### Step 1: Gather Your Ingredients
- **Tea Leaves**: Fresh or dried leaves from your preferred variety.
- **Water**: A sufficient amount of water for the desired strength and flavor.
- **Tea Infusion Equipment**: A teapot, kettle, or pot with a lid.
- **Tea Set (Optional)**: For steeping longer periods.

### Step 2: Prepare Your Environment
-
finish_reason: length


## Prompt formats: the same task, written differently

Task: label movie reviews (Rotten Tomatoes) as **positive** or **negative**. Multiple ways of writing the prompt on 100 test reviews.

In [16]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")
LABELS = ["negative", "positive"]
test = ds["test"].shuffle(seed=42).select(range(100))
texts, gold = test["text"], [LABELS[y] for y in test["label"]]

# 4 examples for few-shot prompts — always from TRAIN, never from test
train = ds["train"].shuffle(seed=1)
examples = [train[i] for i in range(5) if train[i]["label"] == 0][:2] + \
           [train[i] for i in range(5) if train[i]["label"] == 1][:2]

DEFINITIONS = ("You label movie reviews.\n"
               "positive = the reviewer's overall opinion is favourable\n"
               "negative = the reviewer's overall opinion is unfavourable\n"
               "Answer with exactly one word: positive or negative.")

README.md:   0%|          | 0.00/7.46k [00:00<?, ?B/s]

train.parquet: reconstructing file:   0%|          |  0.00B /  699kB            

train.parquet: downloading bytes:           |  0.00B            

validation.parquet: reconstructing file:   0%|          |  0.00B / 90.0kB            

validation.parquet: downloading bytes:           |  0.00B            

test.parquet: reconstructing file:   0%|          |  0.00B / 92.2kB            

test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

In [17]:
def plain(text):                      # 1. one user message, no system prompt
    return [{"role": "user", "content": f"Is this movie review positive or negative?\n{text}"}]

def with_system(text):                # 2. instructions and label definitions in the system prompt
    return [{"role": "system", "content": DEFINITIONS},
            {"role": "user", "content": text}]

def fewshot_turns(text):              # 3. examples as earlier conversation turns
    msgs = [{"role": "system", "content": DEFINITIONS}]
    for ex in examples:
        msgs += [{"role": "user", "content": ex["text"]},
                 {"role": "assistant", "content": LABELS[ex["label"]]}]
    return msgs + [{"role": "user", "content": text}]

def fewshot_block(text):              # 4. examples inside one message, completion style
    shots = "\n\n".join(f"Review: {ex['text']}\nSentiment: {LABELS[ex['label']]}" for ex in examples)
    return [{"role": "user", "content": f"{DEFINITIONS}\n\n{shots}\n\nReview: {text}\nSentiment:"}]

for f in [plain, fewshot_turns, fewshot_block]:
    print(f"===== {f.__name__}")
    print(json.dumps(f("A dull, lifeless film."), indent=1)[:900], "\n")

===== plain
[
 {
  "role": "user",
  "content": "Is this movie review positive or negative?\nA dull, lifeless film."
 }
] 

===== fewshot_turns
[
 {
  "role": "system",
  "content": "You label movie reviews.\npositive = the reviewer's overall opinion is favourable\nnegative = the reviewer's overall opinion is unfavourable\nAnswer with exactly one word: positive or negative."
 },
 {
  "role": "user",
  "content": "there's too much forced drama in this wildly uneven movie , about a young man's battle with his inescapable past and uncertain future in a very shapable but largely unfulfilling present ."
 },
 {
  "role": "assistant",
  "content": "negative"
 },
 {
  "role": "user",
  "content": "astonishing isn't the word -- neither is incompetent , incoherent or just plain crap . indeed , none of these words really gets at the very special type of badness that is deuces wild ."
 },
 {
  "role": "assistant",
  "content": "negative"
 },
 {
  "role": "user",
  "content": "hashiguchi vividly ca

**Structured output**: the server only allows tokens that keep the answer valid JSON with one of the allowed labels (constrained decoding).

In [18]:
SCHEMA = {"type": "json_schema",
          "json_schema": {"name": "sentiment",
                          "schema": {"type": "object",
                                     "properties": {"sentiment": {"type": "string", "enum": LABELS}},
                                     "required": ["sentiment"]}}}

r = client.chat.completions.create(model=VLLM_MODEL, messages=with_system("A dull, lifeless film."),
                                   temperature=0, max_tokens=20, response_format=SCHEMA)
print(r.choices[0].message.content)

{  	"sentiment": "negative"}


In [19]:
from sklearn.metrics import accuracy_score, f1_score

def to_label(answer):
    answer = (answer or "").lower()
    found = [l for l in LABELS if l in answer]
    return found[0] if len(found) == 1 else None          # None = not a valid label

def evaluate(build, max_tokens=10, **kw):
    t0 = time.time()
    preds = []
    for text in texts:
        r = client.chat.completions.create(model=VLLM_MODEL, messages=build(text),
                                           temperature=0, max_tokens=max_tokens, **kw)
        preds.append(to_label(r.choices[0].message.content))
    preds = [p or "invalid" for p in preds]                # an invalid answer counts as wrong
    return {"accuracy": round(accuracy_score(gold, preds), 3),
            "macro_f1": round(f1_score(gold, preds, labels=LABELS, average="macro"), 3),
            "invalid": round(preds.count("invalid") / len(preds), 3),
            "seconds": round(time.time() - t0, 1)}

results = pd.DataFrame({
    "1 plain":            evaluate(plain),
    "2 system prompt":    evaluate(with_system),
    "3 few-shot (turns)": evaluate(fewshot_turns),
    "4 few-shot (block)": evaluate(fewshot_block),
    "5 JSON schema":      evaluate(with_system, response_format=SCHEMA, max_tokens=20),
}).T
results

KeyboardInterrupt: 

In [ ]:
results.to_csv("nb1_results.csv")      # used for comparison in NB3
stop(vllm_proc)                        # free the GPU for Ollama

## Ollama: quantised models, laptop-style

Ollama runs **quantised GGUF models** (≈4-bit) on CPU or GPU — the usual way to run an LLM on a laptop. It also exposes an OpenAI-compatible API, so the client code does not change.

In [21]:
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [22]:
ollama_proc = start_background(["ollama", "serve"], log_file="ollama.log",
                               health_url="http://localhost:11434")

OLLAMA_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"            # alternatives: "gemma3:1b", "llama3.2:1b"
!ollama pull {OLLAMA_MODEL} > /dev/null 2>&1
!ollama list

ready after 5 s
NAME            ID              SIZE      MODIFIED               
qwen2.5:1.5b    65ec06548149    986 MB    Less than a second ago    


In [23]:
# 1) Command line
!ollama run {OLLAMA_MODEL} "Explain in one sentence what a language model does."

A language model is a machine learning system that can generate human-like 
text by predicting the next word in a sequence based on context and previou
previous words.



In [24]:
# 2) Exactly the same Python client as for vLLM — only the URL and the model name change
oclient = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

for review in ["A dull, lifeless film.", "An absolute joy from start to finish."]:
    r = oclient.chat.completions.create(model=OLLAMA_MODEL, messages=with_system(review),
                                        temperature=0, max_tokens=5)
    print(f"{r.choices[0].message.content.strip():>10}  <- {review}")

  negative  <- A dull, lifeless film.
  positive  <- An absolute joy from start to finish.


In [25]:
stop(ollama_proc)

## Which one when?

| | Hugging Face `transformers` | vLLM | Ollama |
|---|---|---|---|
| Runs | inside your Python process | as a server (GPU) | as a server (CPU or GPU) |
| Models | any HF model, full precision | HF models | quantised GGUF models |
| Best for | inspecting models, research, **fine-tuning** (NB3–NB4) | many requests, fast batch evaluation | laptops, demos, offline use |
| Interface | Python objects | OpenAI-compatible API | OpenAI-compatible API + CLI |